In [ ]:
import geojsonimport osimport numpy as np

In [ ]:
ANNOTS_RESIZED = '/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/annotations_resized/'ANNOTS_CLEANED = '/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/annot_cleanned/'

In [ ]:
def load_annotations(path):    """Cargar annotations desde un archivo geojson"""    with open(path, 'r') as f:        return geojson.load(f)def compare_coordinates(coord1, coord2, tolerance=1e-9):    """Comparar dos coordenadas con tolerancia"""    return np.allclose(coord1, coord2, rtol=tolerance, atol=tolerance)def compare_polygons(poly1_coords, poly2_coords, tolerance=1e-9):    """Comparar dos polígonos (lista de coordenadas)"""    if len(poly1_coords) != len(poly2_coords):        return False    for c1, c2 in zip(poly1_coords, poly2_coords):        if not compare_coordinates(c1, c2, tolerance):            return False    return Truedef get_features_by_class(annotations, class_name):    """Obtener todos los features de una clase específica"""    return [f for f in annotations['features'] if f['properties'].get('name') == class_name]

In [ ]:
def compare_slide_annotations(slide_id, tolerance=1e-9):    """Comparar annotations de un slide entre ambas carpetas"""    resized_path = os.path.join(ANNOTS_RESIZED, f'{slide_id}.geojson')    cleaned_path = os.path.join(ANNOTS_CLEANED, f'{slide_id}.geojson')        if not os.path.exists(resized_path) or not os.path.exists(cleaned_path):        return None        resized = load_annotations(resized_path)    cleaned = load_annotations(cleaned_path)        classes = ['Junctional Zone', 'Decidua', 'Labyrinth']    results = {}        for class_name in classes:        resized_features = get_features_by_class(resized, class_name)        cleaned_features = get_features_by_class(cleaned, class_name)                if len(resized_features) != len(cleaned_features):            results[class_name] = {'match': False, 'reason': f'Diferente cantidad: resized={len(resized_features)}, cleaned={len(cleaned_features)}'}            continue                all_match = True        for rf, cf in zip(resized_features, cleaned_features):            rf_coords = rf['geometry']['coordinates']            cf_coords = cf['geometry']['coordinates']                        if not compare_polygons(rf_coords, cf_coords, tolerance):                all_match = False                break                results[class_name] = {'match': all_match, 'count': len(resized_features)}        return results

In [ ]:
slides = [f.replace('.geojson', '') for f in os.listdir(ANNOTS_RESIZED) if f.endswith('.geojson')]print(f"Slides encontrados en annotations_resized: {len(slides)}")print(f"Primeros 5: {sorted(slides)[:5]}")

In [ ]:
results = compare_slide_annotations('slide_001')for class_name, result in results.items():    print(f"{class_name}: {result}")

In [ ]:
all_results = []for slide_id in sorted(slides):    result = compare_slide_annotations(slide_id)    if result:        all_results.append((slide_id, result))        print(f"{slide_id}:", end=" ")        for class_name, r in result.items():            status = "✓" if r['match'] else "✗"            print(f"{class_name}:{status}", end=" ")        print()

In [ ]:
summary = {'match': 0, 'mismatch': 0}for slide_id, result in all_results:    for class_name, r in result.items():        if r['match']:            summary['match'] += 1        else:            summary['mismatch'] += 1print(f"Total comparaciones: {summary['match'] + summary['mismatch']}")print(f"Coinciden: {summary['match']}")print(f"No coinciden: {summary['mismatch']}")

In [ ]:
print("Slides con diferencias:")for slide_id, result in all_results:    for class_name, r in result.items():        if not r['match']:            print(f"  {slide_id} - {class_name}: {r.get('reason', 'Coordenadas diferentes')}")